# Zvec Vector Database Benchmark

Benchmark different Zvec index types (FLAT, HNSW, IVF) with optional quantization on FinDER dataset.

Zvec is an open-source, in-process vector database from Alibaba — lightweight, lightning-fast, no server required.

- **Index Types**: FLAT, HNSW, IVF
- **Quantization**: UNDEFINED (none), FP16, INT8, INT4
- **Metrics**: IP, COSINE, L2

Docs: https://zvec.org/en/ | GitHub: https://github.com/alibaba/zvec

In [1]:
# Install zvec if needed
!pip install zvec

In [2]:
"""
Step 2: Benchmark a specific Zvec index type (NOTEBOOK VERSION)
This version can be run directly in Jupyter notebooks.

Just change the INDEX_TYPE variable below and run the entire script.
Zvec is an in-process DB — no server/Docker needed!
"""

import pickle
import time
import json
import os
import shutil
import pandas as pd
import numpy as np
import zvec
from zvec import (
    CollectionSchema,
    VectorSchema,
    Doc,
    VectorQuery,
    DataType,
    MetricType,
    QuantizeType,
)
from zvec.model.param import (
    FlatIndexParam,
    HnswIndexParam,
    HnswQueryParam,
    IVFIndexParam,
    IVFQueryParam,
)

In [3]:
# ============================================================
# CONFIGURATION — Change INDEX_TYPE here
# ============================================================

INDEX_TYPE = "FLAT"  # Options: FLAT, FLAT_INT8, HNSW, HNSW_INT8, IVF, IVF_INT8

QRELS_PATH = "./icaif-24-finance-rag-challenge/FinDER_qrels.tsv"

# Paths (created by step1 — pre-encoded embeddings)
CORPUS_EMBEDDINGS_PATH = "./encoded_data/corpus_embeddings.pkl"
QUERY_EMBEDDINGS_PATH = "./encoded_data/query_embeddings.pkl"

# Zvec collection storage path
ZVEC_DB_PATH = "./zvec_db"

# Index configurations
INDEX_CONFIGS = {
    "FLAT": {
        "index_param": FlatIndexParam(
            metric_type=MetricType.IP,
            quantize_type=QuantizeType.UNDEFINED,
        ),
        "query_param": None,  # Flat does not need query params
    },
    "FLAT_INT8": {
        "index_param": FlatIndexParam(
            metric_type=MetricType.IP,
            quantize_type=QuantizeType.INT8,
        ),
        "query_param": None,
    },
    "HNSW": {
        "index_param": HnswIndexParam(
            metric_type=MetricType.IP,
            m=16,
            ef_construction=200,
            quantize_type=QuantizeType.UNDEFINED,
        ),
        "query_param": HnswQueryParam(ef=100),
    },
    "HNSW_INT8": {
        "index_param": HnswIndexParam(
            metric_type=MetricType.IP,
            m=16,
            ef_construction=200,
            quantize_type=QuantizeType.INT8,
        ),
        "query_param": HnswQueryParam(ef=100),
    },
    "IVF": {
        "index_param": IVFIndexParam(
            metric_type=MetricType.IP,
            n_list=128,
            quantize_type=QuantizeType.UNDEFINED,
        ),
        "query_param": IVFQueryParam(nprobe=10),
    },
    "IVF_INT8": {
        "index_param": IVFIndexParam(
            metric_type=MetricType.IP,
            n_list=128,
            quantize_type=QuantizeType.INT8,
        ),
        "query_param": IVFQueryParam(nprobe=10),
    },
}

print(f"Selected index type: {INDEX_TYPE}")
print(f"Config: {INDEX_CONFIGS[INDEX_TYPE]}")

Selected index type: FLAT
Config: {'index_param': {"metric_type":IP, "quantize_type":UNDEFINED}, 'query_param': None}


In [4]:
def load_embeddings():
    """Load pre-encoded embeddings"""
    print("\n📥 Loading pre-encoded embeddings...")

    with open(CORPUS_EMBEDDINGS_PATH, "rb") as f:
        corpus_data = pickle.load(f)

    with open(QUERY_EMBEDDINGS_PATH, "rb") as f:
        query_data = pickle.load(f)

    print(f"✅ Loaded:")
    print(f"   Corpus: {corpus_data['embeddings'].shape}")
    print(f"   Queries: {query_data['embeddings'].shape}")

    return corpus_data, query_data


def load_qrels():
    """Load ground truth relevance judgments"""
    df = pd.read_csv(QRELS_PATH, sep="\t")
    qrels = (
        df.groupby("query_id")
        .apply(lambda g: dict(zip(g["corpus_id"], g["score"])))
        .to_dict()
    )
    return qrels

In [5]:
def create_zvec_collection(index_type, embedding_dim=1024):
    """Create a Zvec collection with a specific index type"""

    collection_name = f"finder_{index_type.lower()}"
    collection_path = os.path.join(ZVEC_DB_PATH, collection_name)

    # Remove existing collection directory if it exists
    if os.path.exists(collection_path):
        print(f"🗑️  Removing existing collection: {collection_path}")
        shutil.rmtree(collection_path)

    config = INDEX_CONFIGS[index_type]

    # Define schema with vector index config
    schema = CollectionSchema(
        name=collection_name,
        vectors=VectorSchema(
            name="embedding",
            data_type=DataType.VECTOR_FP32,
            dimension=embedding_dim,
            index_param=config["index_param"],
        ),
    )

    print(f"📦 Creating collection: {collection_name}")
    print(f"   Path: {collection_path}")
    collection = zvec.create_and_open(path=collection_path, schema=schema)

    return collection, collection_path

In [6]:
def insert_corpus(collection, corpus_data, batch_size=1000):
    """Insert corpus embeddings into Zvec"""

    embeddings = corpus_data["embeddings"]
    doc_ids = corpus_data["ids"]

    print(f"\n📤 Inserting {len(doc_ids)} documents into Zvec...")

    insert_start = time.time()

    # Insert in batches
    for i in range(0, len(doc_ids), batch_size):
        batch_ids = doc_ids[i : i + batch_size]
        batch_embeddings = embeddings[i : i + batch_size]

        docs = [
            Doc(
                id=str(doc_id),
                vectors={"embedding": emb.tolist()},
            )
            for doc_id, emb in zip(batch_ids, batch_embeddings)
        ]

        collection.insert(docs)

        if (i + batch_size) % 10000 == 0:
            print(
                f"   Inserted {min(i + batch_size, len(doc_ids))}/{len(doc_ids)} documents"
            )

    insert_time = time.time() - insert_start
    print(f"✅ Inserted all {len(doc_ids)} documents in {insert_time:.2f}s")

    # Flush to ensure data is persisted
    print("💾 Flushing data to disk...")
    collection.flush()

    print(f"✅ Collection stats: {collection.stats}")

    return insert_time

In [7]:
def optimize_collection(collection):
    """Optimize the collection — merges buffered vectors into the configured index"""

    print(f"\n🔨 Optimizing collection (building index)...")

    start_time = time.time()
    collection.optimize()
    build_time = time.time() - start_time

    print(f"✅ Optimization complete in {build_time:.2f}s")
    print(f"   Stats: {collection.stats}")

    return build_time

In [8]:
def search_queries(collection, query_data, index_type, top_k=100):
    """Search queries and measure retrieval time"""

    embeddings = query_data["embeddings"]
    query_ids = query_data["ids"]

    config = INDEX_CONFIGS[index_type]
    query_param = config["query_param"]

    print(f"\n🔍 Searching {len(query_ids)} queries...")
    if query_param:
        print(f"   Query params: {query_param}")

    results = {}
    query_times = []

    for i, (query_id, query_embedding) in enumerate(zip(query_ids, embeddings)):

        start_time = time.time()

        search_result = collection.query(
            vectors=VectorQuery(
                field_name="embedding",
                vector=query_embedding.tolist(),
                param=query_param,
            ),
            topk=top_k,
        )

        query_time = time.time() - start_time
        query_times.append(query_time)

        # Store results — each result is a Doc with id and score
        results[query_id] = {}
        for doc in search_result:
            results[query_id][doc.id] = float(doc.score)

        if (i + 1) % 100 == 0:
            avg_time = np.mean(query_times[-100:])
            print(
                f"   Processed {i + 1}/{len(query_ids)} queries (avg: {avg_time * 1000:.2f}ms/query)"
            )

    total_search_time = sum(query_times)
    avg_query_time = np.mean(query_times)

    print(f"\n✅ Search complete:")
    print(f"   Total time: {total_search_time:.2f}s")
    print(f"   Avg per query: {avg_query_time * 1000:.2f}ms")

    return results, total_search_time, avg_query_time

In [9]:
def calculate_ndcg(qrels, results, k):
    """Calculate NDCG@k"""
    ndcg_scores = []

    for query_id, relevant_docs in qrels.items():
        if query_id not in results:
            continue

        retrieved = results[query_id]
        sorted_docs = sorted(retrieved.items(), key=lambda x: x[1], reverse=True)[:k]

        dcg = 0.0
        for i, (doc_id, score) in enumerate(sorted_docs):
            rel = relevant_docs.get(doc_id, 0)
            dcg += rel / np.log2(i + 2)

        ideal_rels = sorted(relevant_docs.values(), reverse=True)[:k]
        idcg = sum(rel / np.log2(i + 2) for i, rel in enumerate(ideal_rels))

        ndcg = dcg / idcg if idcg > 0 else 0
        ndcg_scores.append(ndcg)

    return np.mean(ndcg_scores)


def calculate_recall(qrels, results, k):
    """Calculate Recall@k"""
    recall_scores = []

    for query_id, relevant_docs in qrels.items():
        if query_id not in results:
            continue

        retrieved = set(list(results[query_id].keys())[:k])
        relevant = set(relevant_docs.keys())

        if len(relevant) > 0:
            recall = len(retrieved & relevant) / len(relevant)
            recall_scores.append(recall)

    return np.mean(recall_scores)


def calculate_precision(qrels, results, k):
    """Calculate Precision@k"""
    precision_scores = []

    for query_id, relevant_docs in qrels.items():
        if query_id not in results:
            continue

        retrieved = list(results[query_id].keys())[:k]
        relevant = set(relevant_docs.keys())

        if len(retrieved) > 0:
            precision = len([doc for doc in retrieved if doc in relevant]) / len(retrieved)
            precision_scores.append(precision)

    return np.mean(precision_scores)


def calculate_map(qrels, results, k):
    """Calculate MAP@k (Mean Average Precision)"""
    ap_scores = []

    for query_id, relevant_docs in qrels.items():
        if query_id not in results:
            continue

        retrieved = list(results[query_id].keys())[:k]
        relevant = set(relevant_docs.keys())

        if len(relevant) == 0:
            continue

        precision_sum = 0.0
        num_relevant = 0

        for i, doc_id in enumerate(retrieved):
            if doc_id in relevant:
                num_relevant += 1
                precision_at_i = num_relevant / (i + 1)
                precision_sum += precision_at_i

        ap = precision_sum / len(relevant) if len(relevant) > 0 else 0
        ap_scores.append(ap)

    return np.mean(ap_scores)


def evaluate(qrels, results):
    """Evaluate retrieval results with comprehensive metrics"""
    metrics = {
        "ndcg@1": calculate_ndcg(qrels, results, 1),
        "ndcg@5": calculate_ndcg(qrels, results, 5),
        "ndcg@10": calculate_ndcg(qrels, results, 10),
        "recall@1": calculate_recall(qrels, results, 1),
        "recall@5": calculate_recall(qrels, results, 5),
        "recall@10": calculate_recall(qrels, results, 10),
        "precision@1": calculate_precision(qrels, results, 1),
        "precision@5": calculate_precision(qrels, results, 5),
        "precision@10": calculate_precision(qrels, results, 10),
        "map@10": calculate_map(qrels, results, 10),
    }

    return metrics

In [10]:
def main():
    index_type = INDEX_TYPE

    print("=" * 60)
    print(f"Benchmarking Zvec Index: {index_type}")
    print("=" * 60)

    # Load data
    corpus_data, query_data = load_embeddings()
    qrels = load_qrels()

    # Zvec is in-process — no server connection needed!
    print(f"\n📦 Using Zvec (in-process vector database)")
    print(f"   DB path: {ZVEC_DB_PATH}")
    os.makedirs(ZVEC_DB_PATH, exist_ok=True)

    try:
        # Create collection with configured index
        collection, collection_path = create_zvec_collection(index_type)

        # Insert corpus
        insert_time = insert_corpus(collection, corpus_data)

        # Optimize — builds the configured index from buffered vectors
        build_time = optimize_collection(collection)

        # Search queries
        results, search_time, avg_query_time = search_queries(
            collection, query_data, index_type
        )

        # Evaluate
        metrics = evaluate(qrels, results)

        # Save results
        output = {
            "vector_db": "zvec",
            "index_type": index_type,
            "insert_time_sec": round(insert_time, 2),
            "index_build_time_sec": round(build_time, 2),
            "total_search_time_sec": round(search_time, 2),
            "avg_query_time_ms": round(avg_query_time * 1000, 2),
            **{k: round(v, 4) for k, v in metrics.items()},
        }

        os.makedirs("./results", exist_ok=True)
        output_file = f"./results/zvec_results_{index_type.lower()}.json"
        with open(output_file, "w") as f:
            json.dump(output, f, indent=2)

        print(f"\n💾 Results saved to: {output_file}")

        return output

    finally:
        # Cleanup — destroy collection to free disk space
        try:
            collection.destroy()
            print("\n🗑️  Collection destroyed")
        except Exception:
            pass


results = main()

Benchmarking Zvec Index: FLAT

📥 Loading pre-encoded embeddings...
✅ Loaded:
   Corpus: (13867, 1024)
   Queries: (216, 1024)

📦 Using Zvec (in-process vector database)
   DB path: ./zvec_db
📦 Creating collection: finder_flat
   Path: ./zvec_db/finder_flat

📤 Inserting 13867 documents into Zvec...
   Inserted 10000/13867 documents
✅ Inserted all 13867 documents in 0.48s
💾 Flushing data to disk...
✅ Collection stats: {"doc_count":13863, "index_completeness":{"embedding":1.000000}}

🔨 Optimizing collection (building index)...
✅ Optimization complete in 0.00s
   Stats: {"doc_count":13863, "index_completeness":{"embedding":1.000000}}

🔍 Searching 216 queries...
   Processed 100/216 queries (avg: 1.45ms/query)
   Processed 200/216 queries (avg: 1.36ms/query)

✅ Search complete:
   Total time: 0.30s
   Avg per query: 1.40ms

💾 Results saved to: ./results/zvec_results_flat.json

🗑️  Collection destroyed


In [11]:
# Display results as a DataFrame
if results:
    df = pd.DataFrame([results])
    display(df)

,vector_db,index_type,insert_time_sec,index_build_time_sec,total_search_time_sec,avg_query_time_ms,ndcg@1,ndcg@5,ndcg@10,recall@1,recall@5,recall@10,precision@1,precision@5,precision@10,map@10
0,zvec,FLAT,0.48,0.0,0.3,1.4,0.3125,0.386,0.4281,0.2531,0.4594,0.5742,0.3125,0.1219,0.0797,0.3685


---
## Run All Index Types and Compare

In [12]:
def benchmark_all_index_types():
    """Run benchmarks for all Zvec index types and compare results"""

    all_results = []
    index_types = ["FLAT", "FLAT_INT8", "HNSW", "HNSW_INT8", "IVF", "IVF_INT8"]

    # Load data once
    corpus_data, query_data = load_embeddings()
    qrels = load_qrels()
    os.makedirs(ZVEC_DB_PATH, exist_ok=True)

    for idx_type in index_types:
        print("\n" + "=" * 60)
        print(f"Benchmarking Zvec Index: {idx_type}")
        print("=" * 60)

        try:
            # Create collection
            collection_name = f"finder_{idx_type.lower()}"
            collection_path = os.path.join(ZVEC_DB_PATH, collection_name)

            if os.path.exists(collection_path):
                shutil.rmtree(collection_path)

            config = INDEX_CONFIGS[idx_type]
            embedding_dim = corpus_data["embeddings"].shape[1]

            schema = CollectionSchema(
                name=collection_name,
                vectors=VectorSchema(
                    name="embedding",
                    data_type=DataType.VECTOR_FP32,
                    dimension=embedding_dim,
                    index_param=config["index_param"],
                ),
            )

            collection = zvec.create_and_open(path=collection_path, schema=schema)

            # Insert
            insert_time = insert_corpus(collection, corpus_data)

            # Optimize (build index)
            build_time = optimize_collection(collection)

            # Search
            search_results, search_time, avg_query_time = search_queries(
                collection, query_data, idx_type
            )

            # Evaluate
            metrics = evaluate(qrels, search_results)

            output = {
                "vector_db": "zvec",
                "index_type": idx_type,
                "insert_time_sec": round(insert_time, 2),
                "index_build_time_sec": round(build_time, 2),
                "total_search_time_sec": round(search_time, 2),
                "avg_query_time_ms": round(avg_query_time * 1000, 2),
                **{k: round(v, 4) for k, v in metrics.items()},
            }

            all_results.append(output)

            # Save individual result
            os.makedirs("./results", exist_ok=True)
            with open(f"./results/zvec_results_{idx_type.lower()}.json", "w") as f:
                json.dump(output, f, indent=2)

        except Exception as e:
            print(f"❌ Error benchmarking {idx_type}: {e}")
            import traceback
            traceback.print_exc()

        finally:
            try:
                collection.destroy()
            except Exception:
                pass

    # Save combined results
    if all_results:
        with open("./results/zvec_combined_results.json", "w") as f:
            json.dump(all_results, f, indent=2)
        print(f"\n💾 Combined results saved to: ./results/zvec_combined_results.json")

    return all_results


# Uncomment to run all benchmarks:
# all_results = benchmark_all_index_types()

In [13]:
# Compare all results
# all_results = benchmark_all_index_types()
# if all_results:
#     df_all = pd.DataFrame(all_results)
#     display(df_all)
#
#     # Performance comparison
#     print("\n📊 Performance Comparison:")
#     print(df_all[["index_type", "index_build_time_sec", "avg_query_time_ms",
#                    "ndcg@10", "recall@10"]].to_string(index=False))